In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = C:\Users\mgmorenogonz\OneDrive - Université Paris 1 Panthéon-Sorbonne\FTD\deeplearning\climate_tail_risk_project


In [4]:
import pandas as pd
from src.paths import PROCESSED

print("PROCESSED =", PROCESSED)
print("Exists:", PROCESSED.exists())
print("Files:", [p.name for p in PROCESSED.iterdir()])

PROCESSED = C:\Users\mgmorenogonz\OneDrive - Université Paris 1 Panthéon-Sorbonne\FTD\deeplearning\climate_tail_risk_project\data\processed
Exists: True
Files: ['article_level_daily', 'news', 'news_daily_panel_v2.csv.csv', 'wti_daily_panel.parquet']


In [6]:
wti = pd.read_parquet(PROCESSED / "wti_daily_panel.parquet")
news = pd.read_csv(PROCESSED / "news_daily_panel_v2.csv")

print("WTI shape:", wti.shape)
print("News shape:", news.shape)
print(news.head())
print(news.columns.tolist())

WTI shape: (2194, 17)
News shape: (3198, 17)
    news_date  n_articles  avg_tone  avg_positivity  avg_negativity  \
0  2016-03-30        5466 -0.425801        2.210297        2.270490   
1  2016-03-31        6844 -0.874777        2.062647        2.708640   
2  2016-04-01        6700 -0.901706        2.104075        2.693165   
3  2016-04-02        3646 -0.682310        1.839107        2.375551   
4  2016-04-03        3099 -0.804668        1.821211        2.460101   

   n_climate  n_carbon  n_cleantech  n_energy  n_policy  n_transition  \
0       5466       199          850      1926      3622          1926   
1       6844       358          896      2124      4485          2124   
2       6700       194          784      2023      4549          2023   
3       3646       109          455       896      2546           896   
4       3099       102          509      1130      2114          1130   

   share_climate  share_carbon  share_cleantech  share_energy  share_policy  \
0         

In [7]:
news = news.copy()
news["news_date"] = pd.to_datetime(news["news_date"], errors="coerce")
news["day"] = news["news_date"].dt.strftime("%Y-%m-%d")

print("Duplicate day rows:", news["day"].duplicated().sum())
print("Coverage:", news["day"].min(), "to", news["day"].max())
print(news.isna().sum().sort_values(ascending=False).head(20))

Duplicate day rows: 0
Coverage: 2016-03-30 to 2024-12-31
news_date           0
n_articles          0
avg_tone            0
avg_positivity      0
avg_negativity      0
n_climate           0
n_carbon            0
n_cleantech         0
n_energy            0
n_policy            0
n_transition        0
share_climate       0
share_carbon        0
share_cleantech     0
share_energy        0
share_policy        0
share_transition    0
day                 0
dtype: int64


In [8]:
merged = wti.merge(
    news.drop(columns=["news_date"]),
    on="day",
    how="left",
    validate="one_to_one"
)

print("WTI shape:", wti.shape)
print("Merged shape:", merged.shape)

cols_to_show = [
    c for c in [
        "date", "day", "wti_price",
        "n_articles", "avg_tone", "avg_positivity", "avg_negativity",
        "n_carbon", "n_cleantech", "n_energy",
        "n_policy", "n_transition",
        "share_carbon", "share_cleantech",
        "share_energy", "share_policy", "share_transition"
    ] if c in merged.columns
]

print(merged[cols_to_show].head(15))

WTI shape: (2194, 17)
Merged shape: (2194, 33)
         date         day  wti_price  n_articles  avg_tone  avg_positivity  \
0  2016-03-30  2016-03-30      36.91      5466.0 -0.425801        2.210297   
1  2016-03-31  2016-03-31      36.94      6844.0 -0.874777        2.062647   
2  2016-04-01  2016-04-01      35.36      6700.0 -0.901706        2.104075   
3  2016-04-04  2016-04-04      34.30      5440.0 -0.848358        2.001062   
4  2016-04-05  2016-04-05      34.52      5697.0 -1.026037        1.996563   
5  2016-04-06  2016-04-06      37.74      6081.0 -0.880441        2.220905   
6  2016-04-07  2016-04-07      37.30      7161.0 -1.012659        2.331964   
7  2016-04-08  2016-04-08      39.74      6201.0 -0.537240        2.159053   
8  2016-04-11  2016-04-11      40.46      5657.0 -0.805963        2.122373   
9  2016-04-12  2016-04-12      42.12      6764.0 -0.795907        2.453877   
10 2016-04-13  2016-04-13      41.70      6800.0 -0.694818        2.689861   
11 2016-04-14  20

In [9]:
news_feature_cols = [
    c for c in [
        "n_articles", "avg_tone", "avg_positivity", "avg_negativity",
        "n_carbon", "n_cleantech", "n_energy", "n_policy", "n_transition",
        "share_carbon", "share_cleantech", "share_energy",
        "share_policy", "share_transition"
    ] if c in merged.columns
]

print(merged[news_feature_cols].isna().mean().sort_values())

n_articles          0.000456
avg_tone            0.000456
avg_positivity      0.000456
avg_negativity      0.000456
n_carbon            0.000456
n_cleantech         0.000456
n_energy            0.000456
n_policy            0.000456
n_transition        0.000456
share_carbon        0.000456
share_cleantech     0.000456
share_energy        0.000456
share_policy        0.000456
share_transition    0.000456
dtype: float64


In [10]:
df = merged.copy()

count_cols = [c for c in [
    "n_articles", "n_carbon", "n_cleantech",
    "n_energy", "n_policy", "n_transition"
] if c in df.columns]

share_cols = [c for c in [
    "share_carbon", "share_cleantech",
    "share_energy", "share_policy", "share_transition"
] if c in df.columns]

tone_cols = [c for c in [
    "avg_tone", "avg_positivity", "avg_negativity"
] if c in df.columns]

# Fill missing news values
for c in count_cols:
    df[c] = df[c].fillna(0)

df["no_news_day"] = df["n_articles"].eq(0).astype(int)

for c in share_cols:
    df[c] = df[c].fillna(0)

for c in tone_cols:
    df[c] = df[c].fillna(0)

# Keep only rows usable for supervised modeling
df_model = df[df["usable_for_model"]].copy()

market_features = [
    "ret_lag1", "abs_ret_lag1", "rv_5", "rv_10", "rv_22",
    "ret_mean_5", "ret_mean_10"
]

news_features = count_cols + share_cols + tone_cols + ["no_news_day"]

model_cols = [
    "day", "date",
    "ret_t_plus_1", "target_tail_t_plus_1"
] + market_features + news_features

df_model = df_model[model_cols].copy()

df_model.to_parquet(PROCESSED / "model_dataset_v1.parquet", index=False)

print("Model dataset shape:", df_model.shape)
print(df_model.head())
print(df_model.isna().sum().sort_values(ascending=False).head(20))
print("Target mean overall:", df_model["target_tail_t_plus_1"].mean())
print("Train target mean:", df_model.loc[df_model["date"] <= "2021-12-31", "target_tail_t_plus_1"].mean())
print("Test target mean:", df_model.loc[df_model["date"] > "2021-12-31", "target_tail_t_plus_1"].mean())

Model dataset shape: (2145, 26)
           day       date  ret_t_plus_1  target_tail_t_plus_1  ret_lag1  \
23  2016-05-02 2016-05-02     -0.024888                     0 -0.001087   
24  2016-05-03 2016-05-03      0.002745                     0 -0.027115   
25  2016-05-04 2016-05-04      0.012713                     0 -0.024888   
26  2016-05-05 2016-05-05      0.005624                     0  0.002745   
27  2016-05-06 2016-05-06     -0.025674                     0  0.012713   

    abs_ret_lag1      rv_5     rv_10     rv_22  ret_mean_5  ...  n_transition  \
23      0.001087  0.032662  0.027623  0.033852    0.014521  ...        1625.0   
24      0.027115  0.033064  0.029051  0.034725    0.014262  ...        2457.0   
25      0.024888  0.036959  0.030541  0.033585    0.005246  ...        2052.0   
26      0.002745  0.018661  0.027558  0.032428   -0.006828  ...        3030.0   
27      0.012713  0.017619  0.027632  0.032412   -0.007526  ...        2063.0   

    share_carbon  share_cleant

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd

df_model = pd.read_parquet(PROCESSED / "model_dataset_v1.parquet")
df_model["date"] = pd.to_datetime(df_model["date"])

train = df_model[df_model["date"] <= "2021-12-31"].copy()
test = df_model[df_model["date"] > "2021-12-31"].copy()

market_features = [
    "ret_lag1", "abs_ret_lag1", "rv_5", "rv_10", "rv_22",
    "ret_mean_5", "ret_mean_10"
]

news_features = [c for c in df_model.columns if c not in [
    "day", "date", "ret_t_plus_1", "target_tail_t_plus_1"
] + market_features]

X_train_m = train[market_features]
X_test_m = test[market_features]

X_train_mn = train[market_features + news_features]
X_test_mn = test[market_features + news_features]

y_train = train["target_tail_t_plus_1"]
y_test = test["target_tail_t_plus_1"]

def fit_eval_logit(X_train, X_test, y_train, y_test, name):
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        ))
    ])
    
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]

    return {
        "model": name,
        "roc_auc": roc_auc_score(y_test, proba),
        "pr_auc": average_precision_score(y_test, proba),
        "test_base_rate": y_test.mean()
    }

results = pd.DataFrame([
    fit_eval_logit(X_train_m, X_test_m, y_train, y_test, "market_only_logit"),
    fit_eval_logit(X_train_mn, X_test_mn, y_train, y_test, "market_plus_news_logit")
])

print(results)

ModuleNotFoundError: No module named 'sklearn'